In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 6
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 6
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [4]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \
    --lookback 30 \
    --horizon 5 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99 \
    #--force


🧹 Запуск модуля очистки данных (Сплиты, Иглы, Выбросы)...
Корректировка сплитов:  10%|██▏                  | 7/68 [00:00<00:01, 60.02it/s]  🕵️‍♂️ [HEURISTIC SPLIT] LSNGP@MISX на 2005-08-03: Коэфф 5.0
  📌 [KNOWN SPLIT] TRNFP@MISX на 2024-02-21: Коэфф 0.01
Корректировка сплитов: 100%|████████████████████| 68/68 [00:00<00:00, 76.49it/s]
✅ Очистка завершена!

🔄 [fold_2010/train - Build] Запуск...
✅ Готово: сохранено в /home/restorator/trader_test/data/processed/2000_2026_1d_30_5/fold_2010/data/train/dataset.csv

🔄 [fold_2010/train - Labels] Запуск...
✅ Авто-уровни: TP=7.90%, SL=7.21%
Разметка: 100%|████████████████████████████████| 39/39 [00:00<00:00, 184.53it/s]
🎉 Размеченный датасет сохранен: labels.csv

🔄 [fold_2010/train - Features] Запуск...

⚙️ [TRAIN] Инициализация расчета...
Этап 1: Расчет кросс-секционных признаков...
Индивидуальные фичи: 100%|██████████████████████| 33/33 [00:44<00:00,  1.34s/it]
✂️ Winsorization (0.1% - 99.9%) для 149 признаков...
📈 Обучение Scaler...
💾 Сохране

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [5]:
!python run_walkforward.py \
    --dataset_dir "data/processed/2000_2026_1d_30_5" \
    --runs 100 \
    --batch_size 8192 \
    --epochs 100 \
    --l2_reg "1e-4" \
    --lr "1e-3" \
    --start_fold "fold_2010" \
    --append

🚀 Запуск массового обучения моделей (Walk-Forward)...
📁 Датасет: data/processed/2000_2026_1d_30_5
⚙️  Настройки: 100 runs, 100 epochs, batch 8192
⏭️ Пропускаем завершенные фолды. Начинаем строго с: fold_2010

🔥 Обучение нейросети для: fold_2010
✅ Mixed precision включена!
✅ Динамическое выделение видеопамяти включено!
🚀 Старт обучения. Фолд: [fold_2010]
📊 Форма данных: [Lookback: 30, Features: 68]
⚙️ Расчет идеальных весов классов...
   Баланс: SL(0)=8805, Hold(1)=22746, TP(2)=8770
   Веса:   SL(0)=1.53, Hold(1)=0.59, TP(2)=1.53
⏳ Подготовка конвейера данных...

--------------------------------------------------
🔄 ИТЕРАЦИЯ 1/100 (Лучшая точность сессии: 0.00%)
--------------------------------------------------
Epoch 1/100
5/5 - 6s - 1s/step - accuracy: 0.4075 - loss: 1.3418 - val_accuracy: 0.4668 - val_loss: 1.1369
Epoch 2/100
5/5 - 1s - 185ms/step - accuracy: 0.4871 - loss: 1.1694 - val_accuracy: 0.5203 - val_loss: 1.0586
Epoch 3/100
5/5 - 1s - 175ms/step - accuracy: 0.5172 - loss: 1.

In [6]:
#очистка наименне успешных ltsm моделей (остается топ 3)
!python -m _tools.clean_lstm_models

🧹 Запуск универсальной уборки моделей (Оставляем Топ-3 по val_loss)...

📂 [2000_2026_1d_30_5] Проверка фолда: fold_2010
  🏆 Оставляем:
     1. Run 16 | Loss: 0.9833 | Acc: 63.56%
     2. Run 48 | Loss: 0.9893 | Acc: 63.68%
     3. Run 57 | Loss: 1.0053 | Acc: 64.86%
  🗑️  Удаляем 4 файлов...

📂 [2000_2026_1d_30_5] Проверка фолда: fold_2012
  🏆 Оставляем:
     1. Run 37 | Loss: 1.0226 | Acc: 59.75%
     2. Run 28 | Loss: 1.0286 | Acc: 59.43%
     3. Run 24 | Loss: 1.0340 | Acc: 59.38%
  🗑️  Удаляем 3 файлов...

📂 [2000_2026_1d_30_5] Проверка фолда: fold_2014
  🏆 Оставляем:
     1. Run 2 | Loss: 1.1622 | Acc: 49.43%
     2. Run 31 | Loss: 1.1703 | Acc: 50.51%
     3. Run 5 | Loss: 1.2446 | Acc: 50.12%
  🗑️  Удаляем 1 файлов...

📂 [2000_2026_1d_30_5] Проверка фолда: fold_2016
  🏆 Оставляем:
     1. Run 8 | Loss: 0.9790 | Acc: 62.87%
     2. Run 1 | Loss: 1.0079 | Acc: 61.75%
     3. Run 5 | Loss: 1.0102 | Acc: 62.28%
  🗑️  Удаляем 1 файлов...

📂 [2000_2026_1d_30_5] Проверка фолда: fold_20

In [ ]:
!python -m _tools.evaluate_lstm_predictions

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.train_rllib_pbt --population 4 --iterations 3000